# §3 Setup — Question Embedding Space (t-SNE)

Visualizes the 134-question v4 sample within the full 1k VQA question pool using
sentence embeddings (all-MiniLM-L6-v2) projected with t-SNE.

**Note:** This cell is slow (~1–2 min) due to embedding 1k questions.

**Data:**
- Pool: `dataset/vqa/vqa1k_semantics.jsonl` (1000 questions)
- Sample: `experiment/s2_v4/s4.csv` (134 questions, v4)

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path

BASE = Path('/home/david/Desktop/yuna/HPA')
sys.path.insert(0, str(BASE / 'analysis'))

# ── Load v4 sample qids ───────────────────────────────────────────────────────
s4 = pd.read_csv(BASE / 'experiment/s2_v4/s4.csv')
samp_qids = set(s4['question_id'].tolist())
print(f'v4 sample: {len(samp_qids)} questions')

# ── Entity/operator grouping maps ────────────────────────────────────────────
ENT_MAP = {
    'object': 'object', 'person': 'person', 'animal': 'animal',
    'food': 'scene',    'place': 'scene',   'vehicle': 'scene',
    'other': 'misc',    'text': 'misc',     'product': 'misc',
}
OP_MAP = {
    'attr': 'attr', 'count': 'count',
    'exist': 'other_op', 'spat': 'other_op', 'ident': 'other_op',
    'act': 'other_op',   'text': 'other_op', 'know': 'other_op',
    'comp': 'other_op',  'cause': 'other_op','temp': 'other_op',
    'other': 'other_op',
}

In [ ]:
# ── Embed & project (slow cell) ───────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE

HF_CACHE = '/home/david/Desktop/yuna/.cache/hf'
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',
                                cache_folder=HF_CACHE)

# Load semantics for the full 1k pool
sem_all = pd.DataFrame(
    [json.loads(l) for l in open(BASE / 'dataset/vqa/vqa1k_semantics.jsonl')]
)[['question_id', 'question', 'op', 'w', 'ent']]

sem_all['ent_grp']   = sem_all['ent'].map(ENT_MAP).fillna('misc')
sem_all['op_grp']    = sem_all['op'].map(OP_MAP).fillna('other_op')
sem_all['is_sample'] = sem_all['question_id'].isin(samp_qids)

print(f'Pool: {len(sem_all)} questions  |  Sample (v4): {sem_all["is_sample"].sum()}')

# Embed
print('Embedding 1k questions...')
embs = embedder.encode(
    sem_all['question'].tolist(),
    batch_size=128, show_progress_bar=True, convert_to_numpy=True
)

# t-SNE
print('Running t-SNE...')
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42)
coords = tsne.fit_transform(embs)
sem_all['x'] = coords[:, 0]
sem_all['y'] = coords[:, 1]
print('Done.')

In [ ]:
# ── Plot 3 panels: ent_grp / op_grp / question_word ─────────────────────────
pool = sem_all[~sem_all['is_sample']]
samp = sem_all[sem_all['is_sample']]

fig, axes = plt.subplots(1, 3, figsize=(19, 6))

panels = [
    ('ent_grp', 'Entity Group',
     {'object': '#e74c3c', 'person': '#3498db', 'animal': '#f39c12',
      'scene':  '#27ae60', 'misc':   '#9b59b6'}),
    ('op_grp', 'Operator Group',
     {'attr': '#e67e22', 'count': '#9b59b6', 'other_op': '#95a5a6'}),
    ('w', 'Question Word',
     {'what': '#e74c3c', 'yesno': '#3498db', 'how_many': '#2ecc71',
      'where': '#e67e22', 'which': '#9b59b6', 'why': '#1abc9c',
      'who': '#f39c12',  'other': '#95a5a6',  'how_long': '#7f8c8d',
      'how_old': '#bdc3c7', 'how': '#34495e'}),
]

for ax, (col, title, cmap) in zip(axes, panels):
    # Pool: small faded dots
    for cat in pool[col].unique():
        color = cmap.get(cat, '#95a5a6')
        sub = pool[pool[col] == cat]
        ax.scatter(sub['x'], sub['y'], c=color, s=6, alpha=0.25, linewidths=0)
    # Sample (v4): larger markers with black edge
    for cat in sorted(samp[col].unique()):
        color = cmap.get(cat, '#95a5a6')
        sub = samp[samp[col] == cat]
        ax.scatter(sub['x'], sub['y'], c=color, s=55, alpha=0.95,
                   edgecolors='k', linewidths=0.5, zorder=5, label=cat)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=7, markerscale=1.2, loc='lower right',
              framealpha=0.8, borderpad=0.4)

# Shared legend: pool vs sample marker size
pool_pt = mlines.Line2D([], [], color='gray', marker='o', linestyle='None',
                         markersize=3, alpha=0.4, label='Full 1k pool')
samp_pt = mlines.Line2D([], [], color='gray', marker='o', linestyle='None',
                         markersize=8, markeredgecolor='k',
                         markeredgewidth=0.5, label=f'v4 sample (N={len(samp)})')
fig.legend(handles=[pool_pt, samp_pt], loc='lower center',
           ncol=2, fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Question Embedding Space (t-SNE, all-MiniLM-L6-v2) — v4 Sample',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()